In [61]:
def generate_summary_table(raw_df):
    # Group by gear position and calculate min/max for vehicle and engine speeds
    summary = raw_df.groupby('transmission_gear_position').agg({
        'vehicle_speed': ['min', 'max'],
        'engine_speed': ['min', 'max']
    }).reset_index()

    # Flatten MultiIndex columns
    summary.columns = ['Gear', 'Min Vehicle Speed', 'Max Vehicle Speed', 'Min Engine Speed', 'Max Engine Speed']

    return summary

def check_data_compliance(raw_df, syn_df):
    max_speed = raw_df['vehicle_speed'].max()
    
    
    def calculate_violations(df):
        total_rows = df.shape[0]

        # Rule 1: Vehicle speed within a reasonable range
        violations_speed = df[(df['vehicle_speed'] < 0) | (df['vehicle_speed'] > max_speed)].shape[0]

        # Rule 2: Reasonable gear changes
        def gear_to_num(gear):
            # Define the mapping from gear position to a numerical value
            mapping = {
                'first': 1,
                'second': 2,
                'third': 3,
                'fourth': 4,
                'fifth': 5,
                'sixth': 6,
                'neutral': 0,
                'reverse': -1,
                'nuetral': 0
            }
            return mapping.get(gear, 0)  # Default to 0 for unknown gears

        df['gear_num'] = df['transmission_gear_position'].apply(gear_to_num)
        df['gear_change'] = df['gear_num'].diff().abs()
        violations_gear = df[df['gear_change'] > 1].shape[0]

        # Rule 3: Matching engine speed, vehicle speed, and gear position
        # Placeholder for now, as specific criteria are needed
        def vs_es_gear_match(syn_df):
            summary_table = generate_summary_table(raw_df)
            lookup_dict = summary_table.set_index('Gear').to_dict('index')

            conditions = [
                (syn_df['transmission_gear_position'] == gear) &
                (syn_df['vehicle_speed'].between(lookup_dict[gear]['Min Vehicle Speed'], lookup_dict[gear]['Max Vehicle Speed'])) &
                (syn_df['engine_speed'].between(lookup_dict[gear]['Min Engine Speed'], lookup_dict[gear]['Max Engine Speed']))
                for gear in lookup_dict
            ]
        
            # Combine conditions
            overall_condition = pd.concat([syn_df[condition] for condition in conditions], axis=0).drop_duplicates()
        
            # Calculate the proportion of violations
            violations_matching = syn_df.shape[0] - overall_condition.shape[0]
        
            return violations_matching

        violations_matching = vs_es_gear_match(df)
        

        # Rule 4: Vehicle speed and engine speed increase/decrease simultaneously
        df['vehicle_speed_change'] = df['vehicle_speed'].diff()
        df['engine_speed_change'] = df['engine_speed'].diff()
        violations_speed_direction = df[(df['vehicle_speed_change'] * df['engine_speed_change'] < 0) & (df['vehicle_speed_change'] != 0) & (df['engine_speed_change'] != 0)].shape[0]

        return {
            'speed_range_violations': f'{violations_speed / total_rows : .2f}',
            'gear_change_violations': f'{violations_gear / total_rows : .2f}',
            'matching_violations': f'{violations_matching / total_rows : .2f}',
            'speed_direction_violations': f'{violations_speed_direction / total_rows : .2f}'
        }

    # Calculate violations for both datasets
    raw_violations = calculate_violations(raw_df)
    syn_violations = calculate_violations(syn_df)

    # return raw_violations, syn_violations

    return syn_violations

In [55]:
import pandas as pd

raw_df = pd.read_csv("../data_selected/openxc/nyc_downtown_east.csv")
syn_df = pd.read_csv("../results/vehiclesec2024/small-scale/csv/realtabformer-tabular_openxc-nyc-downtown-east_20231130183157205073649.csv")

In [56]:
print(check_data_compliance(raw_df, syn_df))

({'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.00', 'matching_violations': ' 0.00', 'speed_direction_violations': ' 0.00'}, {'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.00', 'matching_violations': ' 0.05', 'speed_direction_violations': ' 0.23'})


In [36]:
def summarize_gear_speed_ranges(csv_path):
    # Load the CSV file
    df = pd.read_csv(csv_path)

    # Group by gear position and calculate min/max for vehicle and engine speeds
    summary = df.groupby('transmission_gear_position').agg({
        'vehicle_speed': ['min', 'max'],
        'engine_speed': ['min', 'max']
    }).reset_index()

    # Rename columns for clarity
    summary.columns = ['Gear', 'Min Vehicle Speed', 'Max Vehicle Speed', 'Min Engine Speed', 'Max Engine Speed']

    return summary

summarize_gear_speed_ranges("../data_selected/openxc/nyc_downtown_east.csv")

,Gear,Min Vehicle Speed,Max Vehicle Speed,Min Engine Speed,Max Engine Speed
0,first,0.000000,23.490000,0.0,3076.0
1,fourth,24.539999,49.340000,748.0,3068.0
2,neutral,0.000000,0.000000,766.0,780.0
3,second,11.730000,39.000000,954.0,3406.0
4,third,11.070000,42.969997,732.0,3148.0


In [58]:
import os

CANGen_BASE_FOLDER = '/ocean/projects/cis230033p/yyin4/CANGen'

DICT_DATASET_FILENAME = {
    # OpenXC
    'openxc-nyc-downtown-east': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'nyc_downtown_east.csv'
    ),
    'openxc-india-new-delhi-railway-to-aiims': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'india_New_Delhi_Railway_to_AIIMS.csv'
    ),
    'openxc-taiwan-highwayno2-can': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'taiwan_HighwayNo2_can.csv'
    ),
    'openxc-nyc-downtown-east-no-imputation': os.path.join(
        CANGen_BASE_FOLDER, 'data', 'openxc', 'nyc', 'downtown-east', 'downtown-east_before_imputation.csv'
    ),
    'openxc-india-new-delhi-railway-to-aiims-no-imputation': os.path.join(
        CANGen_BASE_FOLDER, 'data', 'openxc', 'india', 'New_Delhi_Railway_to_AIIMS', 'New_Delhi_Railway_to_AIIMS_before_imputation.csv'
    ),
    'openxc-taiwan-highwayno2-can-no-imputation': os.path.join(
        CANGen_BASE_FOLDER, 'data', 'openxc', 'taiwan', 'HighwayNo2-can', 'HighwayNo2-can_before_imputation.csv'
    ),

    # Car-hacking
    'car-hacking-dos-bits': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'DoS_dataset_aligned_train_bits.csv',
    ),
    'car-hacking-fuzzy-bits': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'Fuzzy_dataset_aligned_train_bits.csv',
    ),
    'car-hacking-rpm-bits': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'RPM_dataset_aligned_train_bits.csv',
    ),
    'car-hacking-gear-bits': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'gear_dataset_aligned_train_bits.csv',
    ),
    'car-hacking-dos-hex': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'DoS_dataset_aligned_train.csv',
    ),
    'car-hacking-fuzzy-hex': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'Fuzzy_dataset_aligned_train.csv',
    ),
    'car-hacking-rpm-hex': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'RPM_dataset_aligned_train.csv',
    ),
    'car-hacking-gear-hex': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'gear_dataset_aligned_train.csv',
    ),

    # SynCAN
    'syncan-raw': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'syncan', 'train.csv'
    )

}

In [62]:
for model in ['realtabformer-tabular', 'ctgan']:
    for dataset in ['openxc-nyc-downtown-east', 'openxc-india-new-delhi-railway-to-aiims', 'openxc-taiwan-highwayno2-can']:
        raw_df = pd.read_csv(DICT_DATASET_FILENAME[dataset])
        res = []
        for csv_file in os.listdir("../results/vehiclesec2024/small-scale/csv"):
            if not csv_file.endswith(".csv"):
                continue
            if model == 'realtabformer-tabular' and (not csv_file.startswith(f"{model}_{dataset}_20231130")):
                continue

            if csv_file.startswith(f"{model}_{dataset}_2023"):
                # print(dataset, model, csv_file)

            
                syn_df = pd.read_csv(os.path.join("../results/vehiclesec2024/small-scale/csv", csv_file))

                print(model, dataset)
                print(check_data_compliance(raw_df, syn_df))

realtabformer-tabular openxc-nyc-downtown-east
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.00', 'matching_violations': ' 0.05', 'speed_direction_violations': ' 0.23'}
realtabformer-tabular openxc-nyc-downtown-east
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.00', 'matching_violations': ' 0.05', 'speed_direction_violations': ' 0.23'}
realtabformer-tabular openxc-nyc-downtown-east
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.00', 'matching_violations': ' 0.05', 'speed_direction_violations': ' 0.23'}
realtabformer-tabular openxc-india-new-delhi-railway-to-aiims
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.05', 'matching_violations': ' 0.00', 'speed_direction_violations': ' 0.22'}
realtabformer-tabular openxc-india-new-delhi-railway-to-aiims
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.05', 'matching_violations': ' 0.00', 'speed_direction_violations': ' 0.22'}
realtabformer-tabular ope